**DATA CLEANING NOTES PER DATASET**

Run `data_ingestion.ipynb` first, then this notebook. Raw → cleaned outputs:

| Source | Raw | Cleaned |
|--------|-----|---------|
| Gemini | `data/raw/ai/gemini_essays_v1.csv` | `data/processed/ai/gemini_essays_v1_cleaned.csv` |
| Claude | `data/raw/ai/claude_dataset.csv` | `data/processed/ai/claude_dataset_cleaned.csv` |
| MGTBench (GPT-3.5) | `data/raw/ai/mgtbench_ai_dataset.csv` | `data/processed/ai/mgtbench_ai_dataset_cleaned.csv` |
| BAWE (human) | `data/raw/human/bawe_dataset.csv` | `data/processed/human/bawe_corpus_dataset_cleaned.csv` |
| **Combined** | (four cleaned files above) | `data/processed/combined_dataset.csv` |

1. **Gemini** — Markdown headings (`## Title`), bold (`**text**`), `*` bullets. Uses `clean_gemini_dataset` (markdown strip + `clean_pipeline`).

2. **Claude** — Curly braces, equations, code. Uses `clean_claude_dataset` + `is_academic_content` filter on prompts.

3. **MGTBench (ChatGPT)** — Heavy math/LaTeX, newlines. Uses `clean_mgtbench_ai_dataset`; one combined CSV (not per-subject files).

4. **BAWE Corpus (human)** — TEI XML ingested in `data_ingestion.ipynb`. Uses `clean_bawe_dataset` (shared `clean_pipeline`).

5. **Combine** — After all four cleaners run, `combine_cleaned_datasets` stacks them into one table (`text`, `label`, `source`, `subject`).

In [1]:
# Import the necessary libraries for cleaning the data
import sys
import importlib
import pandas as pd
from pathlib import Path

# Make src/ importable (works from project root or src/notebooks)
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "data").exists() else CURRENT_DIR.parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Reload so notebook picks up latest cleaning code without kernel restart
import utils.cleaning
for _mod_name in (
    "utils.cleaning.cleaning_methods",
    "utils.cleaning.placeholder_density",
    "utils.cleaning.dataset_cleaning",
    "utils.cleaning",
):
    if _mod_name in sys.modules:
        importlib.reload(sys.modules[_mod_name])

from utils.cleaning import (
    clean_bawe_dataset,
    clean_claude_dataset,
    clean_gemini_dataset,
    clean_mgtbench_ai_dataset,
    combine_cleaned_datasets,
)

pd.set_option('display.max_colwidth', 150)

print("Libraries has been imported!")

Libraries has been imported!


c:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Establish the directories where the data will be read and stored after processing
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "data").exists() else CURRENT_DIR.parent.parent
DATA_DIR = PROJECT_ROOT / "data"

# Directories of the Raw Datasets
RAW_AI_DIR = DATA_DIR / "raw" / "ai"
RAW_HUMAN_DIR = DATA_DIR / "raw" / "human"

# Directories of the Processed Datasets where it will be stored after cleaning the data
PROCESSED_AI_DIR = DATA_DIR / "processed" / "ai"
PROCESSED_HUMAN_DIR = DATA_DIR / "processed" / "human"

# To ensure that the out processed directories exist
PROCESSED_AI_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_HUMAN_DIR.mkdir(parents=True, exist_ok=True)

print("Data Cleaning Paths Ready:")
print(f"  Reading AI Raw Data:        {RAW_AI_DIR.resolve()}")
print(f"  Reading Human Raw Data:     {RAW_HUMAN_DIR.resolve()}")
print(f"  Saving AI Processed Data:   {PROCESSED_AI_DIR.resolve()}")
print(f"  Saving Human Processed Data:{PROCESSED_HUMAN_DIR.resolve()}")


In [ ]:
# Run Claude dataset cleaning
claude_dataset = RAW_AI_DIR / 'claude_dataset.csv'
df_cleaned_claude = clean_claude_dataset(claude_dataset, PROCESSED_AI_DIR, sample_size=None)

In [ ]:
# Run MGTBench AI dataset cleaning 
mgtbench_ai_path = RAW_AI_DIR / 'mgtbench_ai_dataset.csv'
df_cleaned_mgtbench_ai = clean_mgtbench_ai_dataset(mgtbench_ai_path, PROCESSED_AI_DIR, sample_size=None)

In [ ]:
# Run Gemini dataset cleaning
gemini_path = RAW_AI_DIR / 'gemini_essays_v1.csv'
df_cleaned_gemini = clean_gemini_dataset(gemini_path, PROCESSED_AI_DIR, sample_size=None)

In [ ]:
# Run BAWE dataset cleaning (full corpus; set sample_size=N for a quick test)
bawe_path = RAW_HUMAN_DIR / 'bawe_dataset.csv'
df_cleaned_bawe = clean_bawe_dataset(bawe_path, PROCESSED_HUMAN_DIR, sample_size=None)


## Combine cleaned datasets

Stacks BAWE (human, label=0) and MGTBench, Claude, Gemini (AI, label=1) into `data/processed/combined_dataset.csv`.

In [ ]:
# Combine all cleaned datasets (run imports cell first; per-dataset cleaners optional if CSVs already exist)
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "data").exists() else CURRENT_DIR.parent.parent
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_AI_DIR = DATA_DIR / "processed" / "ai"
PROCESSED_HUMAN_DIR = DATA_DIR / "processed" / "human"
PROCESSED_DIR = DATA_DIR / "processed"

df_combined = combine_cleaned_datasets(
    PROCESSED_AI_DIR,
    PROCESSED_HUMAN_DIR,
    output_dir=PROCESSED_DIR,
)


--- COMBINED DATASET SUMMARY ---


,source,label,rows
0,bawe,0,2012
1,claude,1,5083
2,gemini,1,3300
3,mgtbench,1,146609



Total rows: 157,004
Human (label=0): 2,012
AI (label=1):    154,992

Successfully saved combined dataset to:
  C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\combined_dataset.csv

--- SAMPLE COMBINED DATA (FIRST 10 ROWS) ---


,text,label,source,subject
0,Site formation processes are the ways in which artefacts or other types of evidence of past cultures are preserved within the archaeological recor...,0,bawe,Archaeology
1,According to the Encarta Encyclopaedia environmental archaeology by definition 'examines the relationship between human societies and the natural ...,0,bawe,Archaeology
2,"Absolute dating is a very misleading term as it puts across the idea that the method used provides a completely accurate date, which is not true a...",0,bawe,Archaeology
3,"Methods of absolute (or chronometric) dating have developed greatly over recent years, with dendrochronology and radiocarbon dating in particular ...",0,bawe,Archaeology
4,Absolute dating methods have revolutionised geology and archaeology. Within the last 50 years developments in this field have allowed us to create...,0,bawe,Archaeology
5,The dominant theory that Whittle employs in his article is Phenomenology (the study of human experience and consciousness in the every day life) i...,0,bawe,Archaeology
6,This paper is distinctly post - processual in its approach to settlement archaeology and urges the reader to take a broader view of the Neolithic ...,0,bawe,Archaeology
7,Critical Review of Barbara Benders paper Theorising Landscapes and the Prehistoric Landscapes of Stonehenge. Barbara Bender is an archaeological t...,0,bawe,Archaeology
8,"The nature of the transition between Mesolithic and Neolithic periods is a major research concern, in particular how rapidly the change in materia...",0,bawe,Archaeology
9,"Food diary Food descriptive Rice: boiled rice, in especially only used Japanese rice. (Not basmati, long grain rice) Miso soup: miso is a Japanese...",0,bawe,Archaeology
